In [1]:
import sqlalchemy
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import pandas as pd


In [3]:
#Exercise 1 : Open the database
engine = create_engine("sqlite:///chinook.db")
cur = engine.connect()
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

from sqlalchemy.ext.automap import automap_base

Base = automap_base(metadata=metadata)
Base.prepare()

Session = sessionmaker(bind=engine)
session = Session()


In [4]:
#Exercise 2 : Table names
print(metadata.tables.keys())


dict_keys(['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'tracks', 'media_types', 'invoices', 'playlist_track', 'playlists'])


In [5]:
# Exercise 3 : Tracks
Tracks = Base.classes.tracks

query = session.query(Tracks).limit(3)
pd.read_sql(query.statement, engine)


,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99


In [6]:
# Exercise 4 : Albums from Tracks
Albums = Base.classes.albums

query = (
    session.query(
        Tracks.Name.label("Track"),
        Albums.Title.label("Album")
    )
    .join(Albums, Tracks.AlbumId == Albums.AlbumId)
    .limit(20)
)

pd.read_sql(query.statement, engine)


,Track,Album
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Put The Finger On You,For Those About To Rock We Salute You
2,Let's Get It Up,For Those About To Rock We Salute You
3,Inject The Venom,For Those About To Rock We Salute You
4,Snowballed,For Those About To Rock We Salute You
5,Evil Walks,For Those About To Rock We Salute You
6,C.O.D.,For Those About To Rock We Salute You
7,Breaking The Rules,For Those About To Rock We Salute You
8,Night Of The Long Knives,For Those About To Rock We Salute You
9,Spellbound,For Those About To Rock We Salute You


In [7]:
#Exercise 5: Tracks sold
InvoiceItems = Base.classes.invoice_items

query = (
    session.query(
        Tracks.Name.label("Track"),
        InvoiceItems.Quantity
    )
    .join(Tracks, InvoiceItems.TrackId == Tracks.TrackId)
    .limit(10)
)

pd.read_sql(query.statement, engine)


,Track,Quantity
0,Balls to the Wall,1
1,Restless and Wild,1
2,Put The Finger On You,1
3,Inject The Venom,1
4,Evil Walks,1
5,Breaking The Rules,1
6,Dog Eat Dog,1
7,Overdose,1
8,Love In An Elevator,1
9,Janie's Got A Gun,1


In [8]:
#Exercise 6 : Top tracks sold
from sqlalchemy import func

query = (
    session.query(
        Tracks.Name.label("Track"),
        func.sum(InvoiceItems.Quantity).label("Total_Sold")
    )
    .join(InvoiceItems, Tracks.TrackId == InvoiceItems.TrackId)
    .group_by(Tracks.TrackId)
    .order_by(func.sum(InvoiceItems.Quantity).desc())
    .limit(10)
)

pd.read_sql(query.statement, engine)


,Track,Total_Sold
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2
5,Not The Doctor,2
6,Por Causa De Você,2
7,Welcome Home (Sanitarium),2
8,Snowblind,2
9,Cornucopia,2


In [9]:
# Exercise 7 : Top selling artists
Artists = Base.classes.artists

query = (
    session.query(
        Artists.Name.label("Artist"),
        func.sum(InvoiceItems.Quantity).label("Total_Sales")
    )
    .join(Albums, Artists.ArtistId == Albums.ArtistId)
    .join(Tracks, Albums.AlbumId == Tracks.AlbumId)
    .join(InvoiceItems, Tracks.TrackId == InvoiceItems.TrackId)
    .group_by(Artists.ArtistId)
    .order_by(func.sum(InvoiceItems.Quantity).desc())
    .limit(10)
)

pd.read_sql(query.statement, engine)


,Artist,Total_Sales
0,Iron Maiden,140
1,U2,107
2,Metallica,91
3,Led Zeppelin,87
4,Os Paralamas Do Sucesso,45
5,Deep Purple,44
6,Faith No More,42
7,Lost,41
8,Eric Clapton,40
9,R.E.M.,39
